In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.float_format", "{:.3f}".format)

df = pd.read_csv("RIPAoc2020.csv", low_memory=False)

# basic cleaning
df["DATE_OF_STOP"] = pd.to_datetime(df["DATE_OF_STOP"], errors="coerce")
df = df[df["DATE_OF_STOP"].dt.year == 2020].copy()

# ensure stop reason is string
df["REASON_FOR_STOP"] = df["REASON_FOR_STOP"].astype(str).str.upper()

/var/folders/4n/gr221w3n74g83smf7w9pzfy00000gn/T/ipykernel_13695/108664364.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["DATE_OF_STOP"] = pd.to_datetime(df["DATE_OF_STOP"], errors="coerce")


In [2]:
race_cols = {
    "White": "RAE_WHITE",
    "Black": "RAE_BLACK_AFRICAN_AMERICAN",
    "Hispanic": "RAE_HISPANIC_LATINO",
    "Asian": "RAE_ASIAN",
    "Other/Unknown": "RAE_MULTIRACIAL"
}

# ensure numeric
for c in race_cols.values():
    df[c] = pd.to_numeric(df[c], errors="coerce")

def get_race(row):
    for race, col in race_cols.items():
        if row[col] == 1:
            return race
    return np.nan

df["RACE"] = df.apply(get_race, axis=1)
df = df[df["RACE"].notna()].copy()

In [3]:
rs_cols = [
    "RFS_RS_REASON_SUSP",
    "RFS_RS_VIOLENT_CRIME",
    "RFS_RS_DRUG_TRANS",
    "RFS_RS_MATCH_SUSPECT"
]

for c in rs_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["MOVING"] = df["REASON_FOR_STOP"].str.contains("MOVING", na=False)
df["EQUIPMENT"] = df["REASON_FOR_STOP"].str.contains("EQUIP", na=False)

df["INVESTIGATIVE"] = (
    (df["RFS_RS_REASON_SUSP"] == 1) |
    (df["RFS_RS_VIOLENT_CRIME"] == 1) |
    (df["RFS_RS_DRUG_TRANS"] == 1) |
    (df["RFS_RS_MATCH_SUSPECT"] == 1)
)

In [4]:
table1 = df.groupby("RACE").agg(
    total_stops=("RACE", "size"),
    moving=("MOVING", "mean"),
    equipment=("EQUIPMENT", "mean"),
    investigative=("INVESTIGATIVE", "mean")
)

# as percent
table1[["moving", "equipment", "investigative"]] *= 100

# percent of all stops
table1["pct_stops"] = table1["total_stops"] / len(df) * 100

# reorder
table1 = table1[[
    "total_stops",
    "pct_stops",
    "moving",
    "equipment",
    "investigative"
]]

table1.columns = [
    "Total Stops",
    "% of Stops",
    "Moving (%)",
    "Equipment (%)",
    "Investigative (%)"
]

table1

,Total Stops,% of Stops,Moving (%),Equipment (%),Investigative (%)
RACE,,,,,
Asian,2520,6.785,0.000,0.000,2.421
Black,1585,4.268,0.000,0.000,8.833
Hispanic,13612,36.651,0.000,0.000,4.834
White,19423,52.297,0.000,0.000,5.241


In [5]:
total_row = pd.DataFrame({
    "Total Stops": [len(df)],
    "% of Stops": [100.0],
    "Moving (%)": [df["MOVING"].mean() * 100],
    "Equipment (%)": [df["EQUIPMENT"].mean() * 100],
    "Investigative (%)": [df["INVESTIGATIVE"].mean() * 100],
}, index=["Total"])

table1_final = pd.concat([table1, total_row])

table1_final

,Total Stops,% of Stops,Moving (%),Equipment (%),Investigative (%)
Asian,2520,6.785,0.000,0.000,2.421
Black,1585,4.268,0.000,0.000,8.833
Hispanic,13612,36.651,0.000,0.000,4.834
White,19423,52.297,0.000,0.000,5.241
Total,37140,100.000,0.000,0.000,5.054


In [14]:
df_vod = df.copy()
df_vod["TIME_OF_STOP"] = pd.to_numeric(df_vod["TIME_OF_STOP"], errors="coerce")

df_vod = df_vod[df_vod["TIME_OF_STOP"].notna()].copy()

df_vod["HOUR"] = (df_vod["TIME_OF_STOP"] // 100).astype(int)
df_vod["MINUTE"] = (df_vod["TIME_OF_STOP"] % 100).astype(int)

df_vod = df_vod[(df_vod["MINUTE"] < 60) & (df_vod["HOUR"] <= 23)].copy()

In [15]:
df_vod = df_vod[
    (df_vod["HOUR"] >= 17) &
    (df_vod["HOUR"] <= 20)
].copy()

In [16]:
df_vod["BLACK"] = pd.to_numeric(df_vod["RAE_BLACK_AFRICAN_AMERICAN"], errors="coerce").fillna(0)
df_vod["HISP"] = pd.to_numeric(df_vod["RAE_HISPANIC_LATINO"], errors="coerce").fillna(0)

df_vod["MINORITY"] = ((df_vod["BLACK"] == 1) | (df_vod["HISP"] == 1)).astype(int)

In [19]:
df_vod["TIME_BIN"] = df_vod["HOUR"] * 4 + (df_vod["MINUTE"] // 15)

# Veil of Darkness approximation
df_vod["DAYLIGHT"] = (df_vod["HOUR"] < 19).astype(int)

In [20]:
import statsmodels.formula.api as smf

m1 = smf.ols("MINORITY ~ DAYLIGHT + C(TIME_BIN)", data=df_vod).fit(cov_type="HC1")
m2 = smf.ols("BLACK ~ DAYLIGHT + C(TIME_BIN)", data=df_vod).fit(cov_type="HC1")
m3 = smf.ols("HISP ~ DAYLIGHT + C(TIME_BIN)", data=df_vod).fit(cov_type="HC1")

print(len(df_vod))

5212


In [ ]:
# Model outputs
results_table = pd.DataFrame({
    "Model": ["Black or Hispanic", "Black", "Hispanic"],
    "Daylight Coef": [
        m1.params["DAYLIGHT"],
        m2.params["DAYLIGHT"],
        m3.params["DAYLIGHT"]
    ],
    "Std. Error": [
        m1.bse["DAYLIGHT"],
        m2.bse["DAYLIGHT"],
        m3.bse["DAYLIGHT"]
    ],
    "t-Statistic": [
        m1.tvalues["DAYLIGHT"],
        m2.tvalues["DAYLIGHT"],
        m3.tvalues["DAYLIGHT"]
    ],
    "P-Value": [
        m1.pvalues["DAYLIGHT"],
        m2.pvalues["DAYLIGHT"],
        m3.pvalues["DAYLIGHT"]
    ],
    "Observations": [
        int(m1.nobs),
        int(m2.nobs),
        int(m3.nobs)
    ]
})

results_table

,Model,Daylight Coef,Std. Error,t-Statistic,P-Value,Observations
0,Black or Hispanic,0.095,0.024,3.959,0.000,5212
1,Black,0.005,0.010,0.477,0.633,5212
2,Hispanic,0.089,0.024,3.778,0.000,5212


In [30]:
df_out = df.copy()
df_out["ADS_SEARCH_PERSON"] = pd.to_numeric(df_out["ADS_SEARCH_PERSON"], errors="coerce").fillna(0)
df_out["ADS_SEARCH_PROPERTY"] = pd.to_numeric(df_out["ADS_SEARCH_PROPERTY"], errors="coerce").fillna(0)

df_out["ANY_SEARCH"] = ((df_out["ADS_SEARCH_PERSON"] == 1) | (df_out["ADS_SEARCH_PROPERTY"] == 1)).astype(int)

In [31]:
contraband_cols = [
    "CED_FIREARM",
    "CED_AMMUNITION",
    "CED_WEAPON",
    "CED_DRUGS",
    "CED_ALCOHOL",
    "CED_MONEY",
    "CED_DRUG_PARAPHERNALIA",
    "CED_STOLEN_PROP",
    "CED_ELECT_DEVICE",
    "CED_OTHER_CONTRABAND"
]

for c in contraband_cols:
    df_out[c] = pd.to_numeric(df_out[c], errors="coerce").fillna(0)

df_out["CONTRABAND"] = (df_out[contraband_cols].sum(axis=1) > 0).astype(int)

In [34]:
race_cols = {
    "White": "RAE_WHITE",
    "Black": "RAE_BLACK_AFRICAN_AMERICAN",
    "Hispanic": "RAE_HISPANIC_LATINO",
    "Asian": "RAE_ASIAN",
    "Other": "RAE_MULTIRACIAL"
}

for c in race_cols.values():
    df_out[c] = pd.to_numeric(df_out[c], errors="coerce").fillna(0)

def assign_race(row):
    if row["RAE_WHITE"] == 1:
        return "White"
    elif row["RAE_BLACK_AFRICAN_AMERICAN"] == 1:
        return "Black"
    elif row["RAE_HISPANIC_LATINO"] == 1:
        return "Hispanic"
    elif row["RAE_ASIAN"] == 1:
        return "Asian"
    elif row["RAE_MULTIRACIAL"] == 1:
        return "Other"
    else:
        return np.nan

df_out["RACE"] = df_out.apply(assign_race, axis=1)

df_out = df_out[df_out["RACE"].notna()].copy()

In [36]:
df_hit = df_out[df_out["ANY_SEARCH"] == 1].copy()
print(f"Number of hits: {len(df_hit)}")

Number of hits: 8397


In [37]:
model_hit = smf.ols("CONTRABAND ~ C(RACE)", data=df_hit).fit(cov_type="HC1")
model_hit.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             CONTRABAND   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     9.813
Date:                Sun, 26 Apr 2026   Prob (F-statistic):           1.86e-06
Time:                        18:59:12   Log-Likelihood:                -5650.2
No. Observations:                8397   AIC:                         1.131e+04
Df Residuals:                    8393   BIC:                         1.134e+04
Df Model:                           3                                         
Covariance Type:                  HC1                                         
=======================================================================================
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               0.3647      0.030     12.353      0.000       0.307       0.423
C(RACE)[T.Black]       -0.0690      0.038     -1.840      0.066      -0.143       0.004
C(RACE)[T.Hispanic]    -0.0468      0.030     -1.535      0.125      -0.107       0.013
C(RACE)[T.White]        0.0067      0.030      0.219      0.826      -0.053       0.066
==============================================================================
Omnibus:                    45497.045   Durbin-Watson:                   1.912
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             1446.286
Skew:                           0.653   Prob(JB):                         0.00
Kurtosis:                       1.442   Cond. No.                         13.9
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

In [43]:
summary = df_out.groupby("RACE").apply(
    lambda g: pd.Series({
        "Search Rate (%)": g["ANY_SEARCH"].mean() * 100,
        "Hit Rate (%)": g.loc[g["ANY_SEARCH"] == 1, "CONTRABAND"].mean() * 100 if g["ANY_SEARCH"].sum() > 0 else np.nan
    })
).reset_index()

summary

/var/folders/4n/gr221w3n74g83smf7w9pzfy00000gn/T/ipykernel_13695/1214562485.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = df_out.groupby("RACE").apply(


,RACE,Search Rate (%),Hit Rate (%)
0,Asian,10.556,36.466
1,Black,24.543,29.563
2,Hispanic,27.182,31.784
3,White,20.810,37.135


In [44]:
coef_table = pd.DataFrame({
    "Race": model_hit.params.index,
    "Coefficient": model_hit.params.values,
    "Std. Error": model_hit.bse.values,
    "p-value": model_hit.pvalues.values
})

coef_table

,Race,Coefficient,Std. Error,p-value
0,Intercept,0.365,0.030,0.000
1,C(RACE)[T.Black],-0.069,0.038,0.066
2,C(RACE)[T.Hispanic],-0.047,0.030,0.125
3,C(RACE)[T.White],0.007,0.030,0.826


In [47]:
print("Total observations:", len(df_out))
print("Sample size:", len(df_hit))

Total observations: 37140
Sample size: 8397
